In [43]:
import kagglehub

# define model architecture
import torch
import torch.nn as nn
import torch.optim as optim

# split into train and test
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

path = kagglehub.competition_download("playground-series-s4e10")
print(path)


/Users/ignacio/.cache/kagglehub/competitions/playground-series-s4e10


In [44]:
# load the data
import pandas as pd

train_df = pd.read_csv(path + "/train.csv")

# print column names
print(train_df.columns)

Index(['id', 'person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length', 'loan_status'],
      dtype='str')


In [45]:
# ['id', 'person_age', 'person_income', 'person_home_ownership',
#  'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
#  'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file',
#  'cb_person_cred_hist_length', 'loan_status'],

# remove id column
train_df = train_df.drop(columns=["id"])
# change person_home_ownership to categorical with dummy variables and 0,1 for the values
train_df = pd.get_dummies(train_df, columns=["person_home_ownership"], drop_first=True)
# change loan_intent to categorical with dummy variables and 0,1 for the values
train_df = pd.get_dummies(train_df, columns=["loan_intent"], drop_first=True)
# change loan_grade to categorical with dummy variables and 0,1 for the values
train_df = pd.get_dummies(train_df, columns=["loan_grade"], drop_first=True)
# change cb_person_default_on_file to 0,1 for the values
train_df["cb_person_default_on_file"] = train_df["cb_person_default_on_file"].map(
    {"Y": 1, "N": 0}
)

# convert boolean dummy columns to int (pandas get_dummies returns bool in newer versions)
bool_cols = train_df.select_dtypes(include="bool").columns
train_df[bool_cols] = train_df[bool_cols].astype(int)


X = train_df.drop(columns=["loan_status"])
y = train_df["loan_status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# normalize person_age, person_income, person_emp_length, loan_amnt, loan_int_rate, loan_percent_income, cb_person_cred_hist_length to have mean 0 and std 1
scaler = StandardScaler()
X_train[
    [
        "person_age",
        "person_income",
        "person_emp_length",
        "loan_amnt",
        "loan_int_rate",
        "loan_percent_income",
        "cb_person_cred_hist_length",
    ]
] = scaler.fit_transform(
    X_train[
        [
            "person_age",
            "person_income",
            "person_emp_length",
            "loan_amnt",
            "loan_int_rate",
            "loan_percent_income",
            "cb_person_cred_hist_length",
        ]
    ]
)
X_test[
    [
        "person_age",
        "person_income",
        "person_emp_length",
        "loan_amnt",
        "loan_int_rate",
        "loan_percent_income",
        "cb_person_cred_hist_length",
    ]
] = scaler.transform(
    X_test[
        [
            "person_age",
            "person_income",
            "person_emp_length",
            "loan_amnt",
            "loan_int_rate",
            "loan_percent_income",
            "cb_person_cred_hist_length",
        ]
    ]
)


In [46]:
class LoanDefaultNN(nn.Module):
    def __init__(self, input_size):
        super(LoanDefaultNN, self).__init__()
        # fc means fully connected layer
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x


model = LoanDefaultNN(input_size=X_train.shape[1])
print(model)
# define loss function and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

dataloader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=10000,
    shuffle=True,
)
epochs = 400
for epoch in range(epochs):
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()
        y_hat = model(X_batch)
        loss = criterion(y_hat, y_batch)
        loss.backward()
        optimizer.step()
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}", end="\r")


threshold = 0.3

# evaluate the model on the test set
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)
    y_pred = model(X_test_tensor)
    y_pred_label = (y_pred > threshold).float()
    accuracy = (y_pred_label == y_test_tensor).float().mean()
    precision = (y_pred_label * y_test_tensor).sum() / (y_pred_label.sum() + 1e-8)
    recall = (y_pred_label * y_test_tensor).sum() / (y_test_tensor.sum() + 1e-8)
    f1_score = 2 * (precision * recall) / (precision + recall + 1e-8)
    print(f"Test Accuracy: {accuracy.item():.4f}")
    print(f"Test Precision: {precision.item():.4f}")
    print(f"Test Recall: {recall.item():.4f}")
    print(f"Test F1 Score: {f1_score.item():.4f}")


LoanDefaultNN(
  (fc1): Linear(in_features=22, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
)
Test Accuracy: 0.9397: 0.1567
Test Precision: 0.8131
Test Recall: 0.7393
Test F1 Score: 0.7745


In [47]:
test_df = pd.read_csv(path + "/test.csv")
# generate submission with id and loan_status columns
test_df = pd.get_dummies(test_df, columns=["person_home_ownership"], drop_first=True)
test_df = pd.get_dummies(test_df, columns=["loan_intent"], drop_first=True)
test_df = pd.get_dummies(test_df, columns=["loan_grade"], drop_first=True)
test_df["cb_person_default_on_file"] = test_df["cb_person_default_on_file"].map(
    {"Y": 1, "N": 0}
)
bool_cols = test_df.select_dtypes(include="bool").columns
test_df[bool_cols] = test_df[bool_cols].astype(int)
test_df[
    [
        "person_age",
        "person_income",
        "person_emp_length",
        "loan_amnt",
        "loan_int_rate",
        "loan_percent_income",
        "cb_person_cred_hist_length",
    ]
] = scaler.transform(
    test_df[
        [
            "person_age",
            "person_income",
            "person_emp_length",
            "loan_amnt",
            "loan_int_rate",
            "loan_percent_income",
            "cb_person_cred_hist_length",
        ]
    ]
)
test_tensor = torch.tensor(test_df.drop(columns=["id"]).values, dtype=torch.float32)
with torch.no_grad():
    # create new tensor without id column
    test_pred = model(test_tensor)
    test_pred_label = (test_pred > threshold).float().numpy().flatten()
submission_df = pd.DataFrame({"id": test_df["id"], "loan_status": test_pred_label})
submission_df.to_csv("submission_loans.csv", index=False)


In [48]:
!kaggle competitions submit -c playground-series-s4e10 -f submission_loans.csv  -m "Message"

100%|█████████████████████████████████████████| 382k/382k [00:00<00:00, 563kB/s]
Successfully submitted to Loan Approval Prediction